Pretrained LLM
      ↓
4-bit Quantization
      ↓
Frozen 4-bit Base Model
      +
LoRA Adapter
      ↓
Train ONLY LoRA
      ↓
Save LoRA Adapter

In [ ]:
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer


# ============================================================
# STEP 1: DATASET
# ============================================================

data = [
    {
        "text": (
            "### Instruction:\n"
            "What is Python?\n\n"
            "### Response:\n"
            "Python is a high-level programming language used "
            "for web development, data science, AI, and automation."
        )
    },
    {
        "text": (
            "### Instruction:\n"
            "What is RAG?\n\n"
            "### Response:\n"
            "RAG stands for Retrieval-Augmented Generation. "
            "It retrieves relevant information and provides it "
            "to an LLM to generate a better answer."
        )
    },
    {
        "text": (
            "### Instruction:\n"
            "What is a vector database?\n\n"
            "### Response:\n"
            "A vector database stores vector embeddings and "
            "allows similarity search over those embeddings."
        )
    },
    {
        "text": (
            "### Instruction:\n"
            "What is LoRA?\n\n"
            "### Response:\n"
            "LoRA is a parameter-efficient fine-tuning technique "
            "that freezes the base model and trains small "
            "low-rank adapter matrices."
        )
    },
    {
        "text": (
            "### Instruction:\n"
            "What is QLoRA?\n\n"
            "### Response:\n"
            "QLoRA combines LoRA with 4-bit quantization to "
            "reduce GPU memory usage during fine-tuning."
        )
    }
]

dataset = Dataset.from_list(data)

print("Dataset:")
print(dataset)


# ============================================================
# STEP 2: MODEL NAME
# ============================================================

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"


# ============================================================
# STEP 3: LOAD TOKENIZER
# ============================================================

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# STEP 4: CREATE 4-BIT QUANTIZATION CONFIG
# ============================================================

print("\nCreating 4-bit quantization configuration...")

use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

compute_dtype = (
    torch.bfloat16
    if use_bf16
    else torch.float16
)


bnb_config = BitsAndBytesConfig(

    # Load base model using 4-bit
    load_in_4bit=True,

    # NF4 is commonly used for QLoRA
    bnb_4bit_quant_type="nf4",

    # Computation happens in BF16/FP16
    bnb_4bit_compute_dtype=compute_dtype,

    # Additional memory optimization
    bnb_4bit_use_double_quant=True
)


# ============================================================
# STEP 5: LOAD 4-BIT BASE MODEL
# ============================================================

print("\nLoading 4-bit base model...")

model = AutoModelForCausalLM.from_pretrained(

    model_name,

    quantization_config=bnb_config,

    device_map="auto"
)


# ============================================================
# STEP 6: PREPARE QUANTIZED MODEL
# ============================================================

print("\nPreparing model for k-bit training...")

model = prepare_model_for_kbit_training(model)


# ============================================================
# STEP 7: LoRA CONFIGURATION
# ============================================================

print("\nCreating LoRA configuration...")

lora_config = LoraConfig(

    # Rank
    r=16,

    # Scaling factor
    lora_alpha=32,

    # Dropout
    lora_dropout=0.05,

    bias="none",

    # Causal language model
    task_type="CAUSAL_LM",

    # Transformer layers
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)


# ============================================================
# STEP 8: TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    # Output directory
    output_dir="./qlora_model",

    # Number of epochs
    num_train_epochs=3,

    # Small batch because LLMs consume memory
    per_device_train_batch_size=2,

    # Accumulate gradients
    gradient_accumulation_steps=4,

    # Learning rate
    learning_rate=2e-4,

    # Logging
    logging_steps=1,

    # Save checkpoint
    save_steps=50,

    # Keep only latest checkpoints
    save_total_limit=2,

    # Precision
    bf16=use_bf16,

    fp16=torch.cuda.is_available() and not use_bf16,

    # Disable external logging
    report_to="none",

    # Important for some PEFT/QLoRA setups
    remove_unused_columns=False
)


# ============================================================
# STEP 9: CREATE SFT TRAINER
# ============================================================

print("\nCreating trainer...")

trainer = SFTTrainer(

    model=model,

    train_dataset=dataset,

    peft_config=lora_config,

    args=training_args,

    processing_class=tokenizer
)


# ============================================================
# STEP 10: TRAIN
# ============================================================

print("\n===================================")
print("Starting QLoRA training...")
print("===================================\n")

trainer.train()


# ============================================================
# STEP 11: SAVE QLoRA ADAPTER
# ============================================================

print("\nSaving QLoRA adapter...")

trainer.save_model("./qlora_model")

tokenizer.save_pretrained("./qlora_model")


print("\n===================================")
print("QLoRA training completed!")
print("===================================")

print("Adapter saved at:")
print("./qlora_model")